# smDeepFLUOR (separated code)
This notebook runs the same pipeline as `smDeepFLUOR.ipynb` but imports reusable modules from `src/`.

In [ ]:
import os
import sys

# If running from repository root, add it to sys.path
repo_root = os.path.abspath('.')
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from src.cropping import CroppingConfig, crop_trajectories_to_tiff, crops_tiff_to_npz
from src.split_dataset import SplitConfig, split_npz_train_test
from src.data_loading import load_two_classes
from src.standardize import standardize_and_split
from src.train import TrainConfig, train_model
from src.visualization import plot_history
from src.validate import ValidateConfig, load_model_and_validate


## 1) Single-particle cropping (CSV + TIFF -> crops TIFF + NPZ)
Edit paths in the config below.

In [ ]:
cfg = CroppingConfig(
    csv_file_path=r'Imagej_plugin_particle_tracking_output.csv',
    tiff_path=r'TIRF_image.tif',
    output_dir=r'output_folder',
)

# 1) crop to TIFF
written_tiffs = crop_trajectories_to_tiff(cfg)
print('written TIFFs:', len(written_tiffs))

# 2) convert TIFF -> NPZ
written_npz = crops_tiff_to_npz(cfg.output_dir, label_value=0)
print('written NPZ:', len(written_npz))


## 2) Train/test split
Assumes your cropped NPZ files are organized under `root_dir` with subfolders ending in `-crop`.

In [ ]:
split_cfg = SplitConfig(root_dir=r'Root directory where cropped npz files are')
train_dir, test_dir = split_npz_train_test(split_cfg)
print('train_dir:', train_dir)
print('test_dir:', test_dir)


## 3) Data loading (Class A & B)
Point classA/classB to their respective `training` folders.

In [ ]:
classA_train_folder = r'path_to_classA/training'
classB_train_folder = r'path_to_classB/training'

A, B = load_two_classes(classA_train_folder, classB_train_folder, use_fraction=0.7, file_fraction=1.0, seed=0)
print('A:', A.shape, 'B:', B.shape)


## 4) Standardization + split

In [ ]:
X_train, X_val, y_train, y_val = standardize_and_split(A, B)
print(X_train.shape, X_val.shape, y_train.shape, y_val.shape)


## 5) Model training

In [ ]:
train_cfg = TrainConfig(batch_size=32, learning_rate=1e-5, epochs=30, model_dir='./models')
model, history, best_model_path = train_model(X_train, y_train, X_val, y_val, train_cfg=train_cfg)
print('best_model_path:', best_model_path)
plot_history(history)


## 6) Validation on folders

In [ ]:
val_cfg = ValidateConfig(root_dir=r'root_folder where validation data are')
summary = load_model_and_validate(best_model_path, val_cfg)
for item in summary:
    print(item)
